# Lesson 5: Answers from client documents

From [Forward Deployed Engineering](https://gradlabs.in/learn/fde/lessons/lesson5.html). The client, Northstar Skills, is made up: every name, number and message is invented.

Make the agent answer from Northstar's own course pages and policies: cut them into chunks, turn the chunks into embeddings, find the best matches for each question, and answer only from those.

For each cell: **guess what it will print first**, then press ▶ and check.

**1. Get the code.** The second cell installs the small embeddings model's library (Colab usually has it already).

In [ ]:
import os
if not os.path.exists("/content/forward-deployed-engineering"):
    !git clone -q --depth 1 https://github.com/mo-ri-ch/forward-deployed-engineering /content/forward-deployed-engineering
%cd /content/forward-deployed-engineering/starter/northstar-agent
!pip install -q google-genai anthropic openai

In [ ]:
!pip install -q sentence-transformers

**2. Choose a model** for writing answers. `"replay"` needs no key. The embeddings model is separate: it's free and runs inside Colab.

In [ ]:
PROVIDER = "replay"   # "gemini" (free tier), "claude", "openai", or "replay" (no key: recorded answers)

import llm
llm.setup(PROVIDER)

**3. Read Northstar's documents.** Course pages and policies, each with an owner, a date and a status.

🤔 *One of these shouldn't be trusted. Which?*

In [ ]:
import northstar

docs = northstar.load_docs()
for d in docs:
    print(f'{d["file"]:22} {d["status"]:8} updated {d["updated"]}   {len(d["text"].split())} words')

**4. Cut them into chunks.** 50 words each, with 10 words of overlap. Every chunk starts with its page title, so a chunk like "The last 2 months are an internship" still says which course it's about.

In [ ]:
import math


def chunk(doc, size=50, overlap=10):
    words = doc["text"].split()
    pieces = []
    for start in range(0, max(len(words) - overlap, 1), size - overlap):
        piece = " ".join(words[start:start + size])
        pieces.append({"file": doc["file"], "status": doc["status"], "text": f'{doc["title"]}: {piece}'})
    return pieces


chunks = [c for d in docs for c in chunk(d)]
print(len(chunks), "chunks\n")
print(chunks[0]["text"])

🤔 *The Full Stack page: how many chunks does the formula from the lesson predict? Work it out, then run the cell.*

In [ ]:
page = next(d for d in docs if d["file"] == "full-stack.md")
n = len(page["text"].split())
print(n, "words →", len(chunk(page)), "chunks")
print("Formula: (words − o) ÷ (c − o), rounded up =", math.ceil((n - 10) / (50 - 10)))

**5. Turn every chunk into an embedding.** One row of numbers per chunk.

In [ ]:
import embeddings

vectors = embeddings.embed([c["text"] for c in chunks])
print(vectors.shape, "← (chunks, numbers per chunk)")
print(vectors[0][:8].round(3))

**6. Search.** Score every chunk against the question (multiply pair by pair and add up), and keep the top 3.

🤔 *For "hostel for girls?", look at the top score compared with the other two questions.*

In [ ]:
import numpy as np


def search(question, k=3, keep=lambda c: True):
    q = embeddings.embed([question])[0]
    scores = vectors @ q
    order = [i for i in np.argsort(-scores) if keep(chunks[i])][:k]
    return [(float(scores[i]), chunks[i]) for i in order]


for question in ["mern course fees?", "can i pay monthly?", "hostel for girls?"]:
    print("\n" + question)
    for score, c in search(question):
        print(f"  {score:.3f}  {c['file']:20} {c['text'][:60]}…")

Nothing is about hostels, so the closest chunk isn't an answer at all. Nearest isn't the same as relevant.

**7. Check the score by hand.** 🤔 *Work out 0.6 × 0.8 + 0.8 × 0.6 on paper, then run it.*

In [ ]:
a = np.array([0.6, 0.8])
b = np.array([0.8, 0.6])
print(a @ b)

**8. Answer only from what was found.** The rules say: use only these sources, and say so when they don't have the answer.

In [ ]:
ANSWER = """You are the admissions assistant for Northstar Skills.
Answer only from the sources below. If they don't contain the answer, say you don't have that information and offer to have a counsellor message them.
Never promise a job. Reply in under 60 words.

Sources:
{sources}"""


def answer(question, label=None, keep=lambda c: True):
    found = search(question, keep=keep)
    sources = "\n".join(f"[{c['file']}] {c['text']}" for _, c in found)
    print("Q:", question)
    print("Sources:", ", ".join(c["file"] for _, c in found))
    print("Answer:", llm.ask(question, system=ANSWER.format(sources=sources), label=label), "\n")


answer("mern course fees? emi?", "l5-ans-fees")
answer("is hostel available near city centre campus? for girls", "l5-ans-hostel")
answer("if I don't get a job do I get my money back?", "l5-ans-refund")

The hostel answer admits it doesn't know. That's grounding working.

🤔 *Now look at the fee answer and its sources. Is `old-fee-list.md` there?*

**9. Bad documents, bad answers.** The old fee list matches fee questions just as well as the real page. No prompt fixes that. Fix the source: search only documents whose status is `current`.

In [ ]:
def current(c):
    return c["status"] == "current"


for keep, name in [(lambda c: True, "all documents"), (current, "current only")]:
    print(name + ":", [c["file"] for _, c in search("full stack fees", k=4, keep=keep)])

answer("mern course fees? emi?", "l5-ans-fees-fixed", keep=current)

In real life, also ask Arjun to retire the old sheet, so counsellors stop copying from it.

**10. Measure your search.** Eight questions, each with the page that answers it. How often is the right page in the top 3?

In [ ]:
tests = [
    ("mern course fees?", "full-stack.md"),
    ("can i pay in monthly installments?", "fees-and-emi.md"),
    ("any scholarship? we can't afford the full fee", "scholarships.md"),
    ("do you guarantee a job?", "placement.md"),
    ("which campuses do you have?", "campuses.md"),
    ("do I need my own laptop for video editing?", "video-editing.md"),
    ("I want to leave, can I get a refund?", "refunds.md"),
    ("is digital marketing available online?", "digital-marketing.md"),
]


def hit_rate(k=3, show=False):
    hits = 0
    for question, want in tests:
        got = [c["file"] for _, c in search(question, k=k, keep=current)]
        hits += want in got
        if show and want not in got:
            print(f"  missed: {question!r} wanted {want}, got {got}")
    return hits / len(tests)


print(f"Right page in the top 3: {hit_rate(show=True):.0%}")

**11. Your turn: change the chunk size.** 🤔 *Will smaller or bigger chunks find the right page more often?*

In [ ]:
for size in [20, 50, 100]:
    chunks = [c for d in docs for c in chunk(d, size=size, overlap=size // 5)]
    vectors = embeddings.embed([c["text"] for c in chunks])
    print(f"{size:>3} words: {len(chunks):>2} chunks, right page in top 3: {hit_rate():.0%}")

# Put it back to 50 words
chunks = [c for d in docs for c in chunk(d)]
vectors = embeddings.embed([c["text"] for c in chunks])

There's no single right size: it depends on the documents. That's why you measure instead of guessing.

**12. Put it on your GitHub.** Save a copy of this notebook (File → Save a copy in GitHub). Next lesson, the agent stops just answering and starts booking calls.